In [1]:
!pip install xgboost

In [2]:
 pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [3]:
# load pre-trained data
#Imputing and encoding 
#Train Test Split
#Model training And Hypermeter tuning
#Model Evalution & select the best model
#dump the best model

#Imports 
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn import metrics

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

In [4]:
#Load pre
CLEANED_DATA_PATH=r"C:\Users\dshar\OneDrive\Pictures\Desktop\Customer_Churn_prediction\DATA\Clean_data.csv"
data=pd.read_csv(CLEANED_DATA_PATH)
data.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Churn_encoded
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


In [5]:
df=data.copy()

In [6]:
#Feature and Target
X = df.drop(columns=['Churn', 'Churn_encoded'])
y = df['Churn_encoded']

In [7]:
num_cols=X.select_dtypes(include="number").columns.to_list()
cat_cols=X.select_dtypes(include="object").columns.to_list()

In [8]:
from sklearn.compose import ColumnTransformer


In [9]:
#Imputinf and encoding 
num_pipeline=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])
cat_pipeline=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("ohe",OneHotEncoder(handle_unknown="ignore",sparse_output=False,drop="first"))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num_transformer', num_pipeline, num_cols),
        ('cat_transformer', cat_pipeline, cat_cols)
    ]
)


In [10]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [11]:
#Model Dictionary
models={
    "LogisticRegression":(
        LogisticRegression(max_iter=1000),
        {"model__C":[0.01,0.1,1]}
    ),
    "RandomForest":(
        RandomForestClassifier(),
        { "model__n_estimators":[50,100,200],
          "model__max_depth":[None,5,10,20]
        } 
    ),
    "AdaBoost":(
        AdaBoostClassifier(),
        {"model__n_estimators":[50,100,200]}
        
    ),
    "XGBoost":(
        XGBClassifier(eval_metric="logloss" ),
        {"model__n_estimators":[50,100,200],
         "model__learning_rate":[0.01,0.1,1]
        }
    ),
    "SVC":(
        SVC(),
        {
            "model__C":[0.01,0.1,1],
            "model__kernel":["rbf","linear","polynomial"]
        
        }
        
    )
    
        
    
}

In [12]:
import joblib

In [13]:
result=[]
best_model =None
best_score=0
# Training 
for name ,(model,param) in models.items():
    pipe=Pipeline(steps=[
        ("preprocess",preprocessor),
        ("model",model)])
    
grid=GridSearchCV(
    pipe,
    param_grid=param,
    cv=5,
    scoring="f1",
    n_jobs=-1)
grid.fit(X_train,y_train)
y_pred=grid.predict(X_test)
acc=metrics.accuracy_score(y_test,y_pred)
f1=metrics.f1_score(y_test,y_pred)


    
result.append({
    "Model":model,
    "Accuracy":acc,
    "F1_Score":f1
})

if f1>best_score:
    best_score=f1 
    best_model=grid.best_estimator_

result_df=pd.DataFrame(result)  

#sort the dataframe about f1_Score
result_df=result_df.sort_values(by="F1_Score",ascending=False)

print("Model Comparison")
print(result_df)

print("\nBest model:",result_df.iloc[0]["Model"])

MODEL_PATH=r"C:\Users\dshar\OneDrive\Pictures\Desktop\Customer_Churn_prediction\Best_Model.pkl"
#Dump the model
joblib.dump(best_model,MODEL_PATH)
print("Model Trained and Saved sucessfully")
        
        

        
    

C:\Users\dshar\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
15 fits failed out of a total of 45.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
7 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\dshar\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\dshar\anaconda3\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\Users\dshar\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 663, in fit
    self._final_estimator.fit(

Model Comparison
   Model  Accuracy  F1_Score
0  SVC()   0.82115  0.637931

Best model: SVC()
Model Trained and Saved sucessfully
